In [ ]:

%matplotlib inline

from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/scratch_root/ed425/HiRISE_diffusion")]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "data" / "dataset.py").is_file():
            return candidate.resolve()
    raise RuntimeError("Could not locate HiRISE_diffusion project root")


project_root = find_project_root()
for import_path in (project_root, project_root / "src"):
    import_path_str = str(import_path)
    if import_path_str not in sys.path:
        sys.path.insert(0, import_path_str)

from data.dataset import DiffusionDataset
from config import DDPMModelConfig, DDPMInferenceConfig
from models import BidirectionalDDPMUNet
from diffusion.scheduler import DDPMScheduler
from compute_prior import load_prior_stats
from inference_ddpm import sample
from eval_ddpm import _ssim_safe


def find_data_paths():
    candidates = [
        (project_root / "data" / "files" / "data_record_bin12.csv", project_root / "data"),
        (Path("/scratch_root/ed425/HiRISE/files/data_record_bin12.csv"), Path("/scratch_root/ed425/HiRISE")),
        (Path("/scratch_root/as5023/HiRISE/data/data_record_bin12.csv"), Path("/scratch_root/as5023/HiRISE/data")),
    ]
    for csv_path, data_root in candidates:
        if csv_path.is_file() and data_root.is_dir():
            return csv_path, data_root
    searched = "\n".join(f"  csv={csv}  root={root}" for csv, root in candidates)
    raise FileNotFoundError(f"Could not find data_record_bin12.csv/data root. Searched:\n{searched}")


csv_path, data_root = find_data_paths()
output_dir = project_root / "src" / "output"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"project_root = {project_root}")
print(f"csv_path     = {csv_path}")
print(f"data_root    = {data_root}")
print(f"output_dir   = {output_dir}")
print(f"device       = {device}")


In [ ]:

start_time = time.time()
dr = pd.read_csv(csv_path)
allowed_sets = [19645, 7292, 7293, 14774]

dataset = DiffusionDataset(
    data_record=dr,
    data_root=str(data_root),
    sweep=False,
    allowed_sets=allowed_sets,
)

print(f"Dataset initialisation time: {time.time() - start_time:.2f} seconds")


In [ ]:
sample = dataset[2]
print(sample.keys())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(4, 5))

axs[0].imshow(sample['ir'][0].numpy(), cmap='gray')
axs[0].set_title("IR10")
axs[0].axis('off')
axs[1].imshow(sample['red'][0].numpy(), cmap='gray')
axs[1].set_title("RED4 (target)")
axs[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:

cfg_model = DDPMModelConfig()
cfg_inf = DDPMInferenceConfig()

model = BidirectionalDDPMUNet(
    in_channels=cfg_model.in_channels,
    out_channels=cfg_model.out_channels,
    base_channels=cfg_model.base_channels,
    num_res_blocks=cfg_model.num_res_blocks,
    dropout=cfg_model.dropout,
).to(device)

checkpoint_path = output_dir / "latest_bidirectional.pt"
prior_red_path = output_dir / "prior_red.pt"
prior_ir_path = output_dir / "prior_ir.pt"

for required_path in (checkpoint_path, prior_red_path, prior_ir_path):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
loss = ckpt.get("loss", "?")
loss_text = f"{loss:.4f}" if isinstance(loss, (float, int)) else str(loss)
print(f"Loaded checkpoint  step={ckpt.get('step', '?')}  loss={loss_text}")

scheduler = DDPMScheduler(
    timesteps=cfg_model.timesteps,
    beta_start=cfg_model.beta_start,
    beta_end=cfg_model.beta_end,
).to(device)

prior_red_stats = load_prior_stats(str(prior_red_path), device)
prior_ir_stats = load_prior_stats(str(prior_ir_path), device)

print(f"Prior RED: mu={prior_red_stats['mu'].item():.4f}  sigma={prior_red_stats['sigma'].item():.4f}")
print(f"Prior IR:  mu={prior_ir_stats['mu'].item():.4f}  sigma={prior_ir_stats['sigma'].item():.4f}")


In [ ]:
prior_red_hist = prior_red_stats['histogram'].cpu().numpy()
prior_ir_hist = prior_ir_stats['histogram'].cpu().numpy()

bins_red = int(prior_red_stats['bins'].item())
bins_ir  = int(prior_ir_stats['bins'].item())

x_red = np.linspace(prior_red_stats['hist_min'].item(), prior_red_stats['hist_max'].item(), bins_red)
x_ir  = np.linspace(prior_ir_stats['hist_min'].item(), prior_ir_stats['hist_max'].item(), bins_ir)

red_nz = int((prior_red_hist > 1e-6).sum())
ir_nz  = int((prior_ir_hist > 1e-6).sum())

fig, axs = plt.subplots(1, 1, figsize=(10, 3.6))

axs.plot(x_red, prior_red_hist, color='red', lw=1.8, label='RED')
axs.plot(x_ir, prior_ir_hist, color='blue', lw=1.8, label='IR')
axs.set_yscale('log')
axs.set_xlabel('Normalized value')
axs.set_ylabel('Probability (log scale)')
axs.set_title('Prior Histograms (RED vs IR)')
axs.grid(alpha=0.25)
axs.legend(loc='upper right', fontsize=9)


print(f"RED histogram sum={prior_red_hist.sum():.6f}, nonzero(>1e-6) bins={red_nz}/{bins_red}")
print(f"IR  histogram sum={prior_ir_hist.sum():.6f}, nonzero(>1e-6) bins={ir_nz}/{bins_ir}")

In [ ]:
prior_red_stats.keys()

In [ ]:

# Pick a sample from DiffusionDataset and run both inference directions.
SAMPLE_IDX = 2
SEED = 42

diff_sample = dataset[SAMPLE_IDX]
print("obs={}  set={}".format(diff_sample["obs_id"], diff_sample["set_name"]))

norm_stats = diff_sample["norm_stats"]  # [center, scale, dc]
center = norm_stats[0].item()
scale = norm_stats[1].item()
dc = norm_stats[2].item()
print("norm_stats: center={:.4f}  scale={:.4f}  dc={:.4f}".format(center, scale, dc))

use_denorm = False

def denorm(arr):
    """Reverse DiffusionDataset normalization: x_raw = (x_norm + dc) * scale + center."""
    if use_denorm:
        return (np.clip(arr, -10, 10) + dc) * scale + center
    return arr


def seed_torch(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

# Direction 0: IR10 -> RED4
ir_tensor = diff_sample["ir"].unsqueeze(0).to(device)
ir_norm = diff_sample["ir"].squeeze().cpu().numpy()
red_gt_norm = diff_sample["red"].squeeze().cpu().numpy()

seed_torch(SEED)
cfg_dir0 = DDPMInferenceConfig(lambda_scl=100.0, lambda_ccl=100.0)
with torch.no_grad():
    pred = sample(
        model,
        scheduler,
        ir_tensor,
        direction=0,
        prior_stats=prior_red_stats,
        cfg_inf=cfg_dir0,
        device=device,
    )
red_pred_norm = pred.squeeze().cpu().numpy()

ir_display = denorm(ir_norm)
red_pred = denorm(red_pred_norm)
red_gt = denorm(red_gt_norm)
print("[IR->RED]  ir_src={:.4f}  pred={:.4f}  GT={:.4f}".format(
    ir_display.mean(), red_pred.mean(), red_gt.mean()))

# Direction 1: RED4 -> IR10
red_tensor = diff_sample["red"].unsqueeze(0).to(device)
red_norm = diff_sample["red"].squeeze().cpu().numpy()
ir_gt_norm = diff_sample["ir"].squeeze().cpu().numpy()

seed_torch(SEED)
cfg_dir1 = DDPMInferenceConfig(lambda_scl=100.0, lambda_ccl=100.0)
with torch.no_grad():
    pred_ir = sample(
        model,
        scheduler,
        red_tensor,
        direction=1,
        prior_stats=prior_ir_stats,
        cfg_inf=cfg_dir1,
        device=device,
    )
ir_pred_norm = pred_ir.squeeze().cpu().numpy()

red_display = denorm(red_norm)
ir_pred = denorm(ir_pred_norm)
ir_gt = denorm(ir_gt_norm)
print("[RED->IR]  red_src={:.4f}  pred={:.4f}  GT={:.4f}".format(
    red_display.mean(), ir_pred.mean(), ir_gt.mean()))


In [ ]:

fig, axs = plt.subplots(2, 3, figsize=(14, 9))

# Row 0: IR10 -> RED4
axs[0, 0].imshow(ir_display, cmap="gray")
axs[0, 0].set_title("IR10 (source)" + chr(10) + "mean={:.4f}".format(ir_display.mean()))
axs[0, 0].axis("off")

axs[0, 1].imshow(red_pred, cmap="gray")
axs[0, 1].set_title("RED4 (predicted)" + chr(10) + "mean={:.4f}".format(red_pred.mean()))
axs[0, 1].axis("off")

axs[0, 2].imshow(red_gt, cmap="gray")
axs[0, 2].set_title("RED4 (GT)" + chr(10) + "mean={:.4f}".format(red_gt.mean()))
axs[0, 2].axis("off")

# Row 1: RED4 -> IR10
axs[1, 0].imshow(red_display, cmap="gray")
axs[1, 0].set_title("RED4 (source)" + chr(10) + "mean={:.4f}".format(red_display.mean()))
axs[1, 0].axis("off")

axs[1, 1].imshow(ir_pred, cmap="gray")
axs[1, 1].set_title("IR10 (predicted)" + chr(10) + "mean={:.4f}".format(ir_pred.mean()))
axs[1, 1].axis("off")

axs[1, 2].imshow(ir_gt, cmap="gray")
axs[1, 2].set_title("IR10 (GT)" + chr(10) + "mean={:.4f}".format(ir_gt.mean()))
axs[1, 2].axis("off")

fig.text(0.01, 0.73, "IR->RED", va="center", rotation="vertical", fontsize=13, fontweight="bold")
fig.text(0.01, 0.27, "RED->IR", va="center", rotation="vertical", fontsize=13, fontweight="bold")

mse_d0 = ((red_pred - red_gt) ** 2).mean()
mse_d1 = ((ir_pred - ir_gt) ** 2).mean()
space_label = "Physical " if use_denorm else "Normalized "
plt.suptitle(
    "obs={}  set={}  {}MSE(IR->RED)={:.4e}  MSE(RED->IR)={:.4e}".format(
        diff_sample["obs_id"], diff_sample["set_name"], space_label, mse_d0, mse_d1
    ),
    y=1.01,
)
plt.tight_layout()
plt.show()


## Experiment to test different values for lambdas
- Focusing on IR to RED

In [ ]:

# Run inference for each lambda value with a fixed seed for fair comparison.
lambdas = list(range(0, 110, 10))
sci_predictions = {}
SEED = 42

for lam in lambdas:
    seed_torch(SEED)
    cfg = DDPMInferenceConfig(lambda_scl=float(lam), lambda_ccl=float(lam))
    with torch.no_grad():
        pred = sample(
            model,
            scheduler,
            ir_tensor,
            direction=0,
            prior_stats=prior_red_stats,
            cfg_inf=cfg,
            device=device,
            verbose=False,
        )
    sci_predictions[lam] = pred.squeeze().cpu().numpy()
    print(
        f"lambda={lam:3d}  "
        f"range=[{sci_predictions[lam].min():.3f}, {sci_predictions[lam].max():.3f}]  "
        f"mean={sci_predictions[lam].mean():.3f}"
    )


In [ ]:

n_pred = len(lambdas)
fig, axs = plt.subplots(1, n_pred + 2, figsize=(4 * (n_pred + 2), 4))

axs[0].imshow(ir_display, cmap="gray")
axs[0].set_title("IR10 (source)")
axs[0].axis("off")

for i, lam in enumerate(lambdas):
    axs[i + 1].imshow(sci_predictions[lam], cmap="gray")
    mean_val = sci_predictions[lam].mean()
    axs[i + 1].set_title(f"lambda = {lam}\nmean={mean_val:.2f}")
    axs[i + 1].axis("off")

axs[-1].imshow(red_gt, cmap="gray")
axs[-1].set_title(f"GT (RED4)\nmean={red_gt.mean():.2f}")
axs[-1].axis("off")

plt.suptitle(f"SCI weight sweep - obs={diff_sample['obs_id']}  set={diff_sample['set_name']}", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:

# Always derive GT from diff_sample to avoid stale-variable bugs.
red_gt_sweep = diff_sample["red"].squeeze().cpu().numpy()
gt_t = torch.from_numpy(red_gt_sweep)
print(f"SSIM sweep using: obs={diff_sample['obs_id']}  set={diff_sample['set_name']}  GT mean={red_gt_sweep.mean():.3f}")

norm_stats = diff_sample["norm_stats"]
gt_t = (gt_t + norm_stats[2]) * norm_stats[1] + norm_stats[0]

ssim_scores = {}
for lam in lambdas:
    pred_t = torch.from_numpy(sci_predictions[lam])
    pred_t = (pred_t + norm_stats[2]) * norm_stats[1] + norm_stats[0]
    dr = (torch.max(gt_t.max(), pred_t.max()) - torch.min(gt_t.min(), pred_t.min())).clamp(min=0.01).item()
    ssim_scores[lam] = _ssim_safe(
        gt_t.squeeze().cpu().numpy(),
        pred_t.squeeze().cpu().numpy(),
        data_range=dr,
    )
    print(f"lambda={lam:3d}  SSIM={ssim_scores[lam]:.4f}")

best_lam = max(ssim_scores, key=ssim_scores.get)
print(f"\nBest lambda = {best_lam}  (SSIM={ssim_scores[best_lam]:.4f})")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar([str(l) for l in lambdas], list(ssim_scores.values()), color="steelblue")
bars[lambdas.index(best_lam)].set_color("tomato")
ax.set_xlabel("lambda_scl = lambda_ccl")
ax.set_ylabel("SSIM vs Ground Truth")
ax.set_title("SCI weight sweep - SSIM to GT")
for bar, val in zip(bars, ssim_scores.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.show()
